In [1]:
import anthropic
import dotenv
print("Both installed!")

# Load env variables
from dotenv import load_dotenv

load_dotenv()
print("Env variables loaded!")

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"
print("Client created!")

#Helper Functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat (messages, system=None, temperature=0.7, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text
print("Helper functions defined!")

Both installed!
Env variables loaded!
Client created!
Helper functions defined!


In [ ]:
import json


def generate_dataset():
    prompt = """
Create a JSON configuration for an AWS Lambda function that sets up a basic Python runtime
with a memory allocation of 512 MB and a timeout of 10 seconds,
    "format": "json",
    "solution_criteria": "Must include runtime, memory size, and timeout
    and basic structure of a AWS Lambda configuration."

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python/json/regex",
        "solution_criteria": "key criteria for evaluating the solution",
    },
    ...additional
]
```
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation.")
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    return json.loads(text)

In [6]:
dataset = generate_dataset()

with (open("005_task_dataset.json", "w")) as f:
    json.dump(dataset, f, indent=2)
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Create a JSON configuration for an AWS Lambda function that sets up a basic Python runtime with a memory allocation of 512 MB and a timeout of 10 seconds",
    "format": "json",
    "solution_criteria": "Must include runtime, memory size, and timeout and basic structure of a AWS Lambda configuration.",
    "solution": {
      "FunctionName": "MyPythonLambdaFunction",
      "Runtime": "python3.11",
      "Role": "arn:aws:iam::123456789012:role/lambda-execution-role",
      "Handler": "lambda_function.lambda_handler",
      "MemorySize": 512,
      "Timeout": 10,
      "Description": "A basic Python Lambda function with 512MB memory and 10 second timeout",
      "Environment": {
        "Variables": {
          "ENV": "production"
        }
      },
      "Tags": {
        "Project": "MyProject",
        "Environment": "Production"
      },
      "TracingConfig": {
        "Mode": "PassThrough"
      },
      "Architectures": [
        "x86_64"
      ],
      "Ephemera

In [10]:
def run_prompt(test_case):
    """Merges the prompt and the test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON or a plain Regex.
* Do not include any explanations or additional text.
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
import re

def grade_by_model(test_case, output):
    eval_prompt = f"""
Please grade the following output on a scale of 1 to 10, with 10 being the best.
The output should be graded based on how well it solves the task, and how well it follows the instructions.
Include a breif reasoning for your grading.

Original Task:
<task>
{test_case['task']}
</task>

Solution to evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case['solution_criteria']}
</criteria>

Output Format:
Provide your grading in a structured JSON format with the following fields, in this exact order:
- "strengths": A brief description of the strengths of the solution.
- "weaknesses": A brief description of the weaknesses of the solution.
- "reasoning": A brief explanation of the reasoning behind the score.
- "score": A numeric score from 1 to 10, with 10 being the best.

Respond with only the JSON. Keep your response concise and to the point.
Use plain English only in all string values. Do not include code, regex patterns, or backslashes.
Example response:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}

"""

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation. Do not use backslashes in any string values.")
    text = eval_text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    text = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', text)
    return json.loads(text)

In [8]:
import re
import ast

def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    return text

def validate_json(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        parsed = json.loads(text)
        score += 6
        if isinstance(parsed, (dict, list)) and len(parsed) > 0:
            score += 2
        if len(text) > 50:
            score += 2
    except json.JSONDecodeError:
        if text.startswith("{") or text.startswith("["):
            score += 2
    return min(score, 10)

def validate_python(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        tree = ast.parse(text)
        score += 6
        if any(isinstance(n, ast.FunctionDef) for n in ast.walk(tree)):
            score += 2
        if len(text.splitlines()) > 3:
            score += 2
    except SyntaxError:
        if "def " in text or "import " in text:
            score += 2
    return min(score, 10)

def validate_regex(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        re.compile(text)
        score += 6
        if len(text) > 5:
            score += 2
        if any(c in text for c in r"()[]{}^$"):
            score += 2
    except re.error:
        score += 1
    return min(score, 10)

def grade_syntax(response, test_case):
    fmt = test_case['format']
    if fmt == 'regex':
        return validate_regex(response)
    elif fmt == 'json':
        return validate_json(response)
    elif fmt == 'python':
        return validate_python(response)
    else:
        raise ValueError(f"Unknown format: {fmt}")


In [11]:
def run_test_case(test_case):
    """Calls run_prompt with the test case, then evaluates the result and returns a score"""
    output = run_prompt(test_case)

    model_result = grade_by_model(test_case, output)
    model_score = model_result["score"]
    reasoning = model_result["reasoning"]
    strengths = model_result["strengths"]
    weaknesses = model_result["weaknesses"]

    code_score = grade_syntax(output, test_case)

    score = (model_score + code_score) / 2


    return {
        "output": output,
        "test_case": test_case,
        "model score": model_score,
        "code score": code_score,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses
    }

In [12]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score}")

    return results


In [13]:
with  open("005_task_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average Score: 9.5
[
  {
    "output": "```json\n{\n  \"FunctionName\": \"my-lambda-function\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::123456789012:role/lambda-execution-role\",\n  \"Handler\": \"lambda_function.lambda_handler\",\n  \"MemorySize\": 512,\n  \"Timeout\": 10,\n  \"Description\": \"Basic Python Lambda function\",\n  \"Environment\": {\n    \"Variables\": {}\n  },\n  \"TracingConfig\": {\n    \"Mode\": \"PassThrough\"\n  }\n}\n```",
    "test_case": {
      "task": "Create a JSON configuration for an AWS Lambda function that sets up a basic Python runtime with a memory allocation of 512 MB and a timeout of 10 seconds",
      "format": "json",
      "solution_criteria": "Must include runtime, memory size, and timeout and basic structure of a AWS Lambda configuration.",
      "solution": {
        "FunctionName": "MyPythonLambdaFunction",
        "Runtime": "python3.11",
        "Role": "arn:aws:iam::123456789012:role/lambda-execution-role",
        "Hand